# GPT-2 model: build, load pretrained weights, train

**Key learnings**

1. `model.named_parameters()` vs `model.state_dict()`:  
   - `named_parameters()` yields only non-redundant (leaf) parameters — one fewer entry when `lm_head.weight` is tied to `wte.weight`  
   - `state_dict()` includes every registered buffer/weight

2. `self.apply(self._init_weights)` recursively calls the init function on every submodule

3. When copying pretrained weights, use `dst.copy_(src)` — **not** `dst = src`.  
   `copy_()` writes into the existing tensor in-place, keeping all references intact.

4. Any operation that touches a weight tensor during weight loading must be wrapped in `torch.no_grad()` to avoid polluting the computation graph.

5. Only 2-D weight matrices get L2 / weight-decay; biases and LayerNorm parameters do not (they have no "direction" to shrink).

6. `grad_accum_steps` lets you simulate a large batch (`n_tokens_per_batch`) by accumulating gradients over multiple smaller forward/backward passes before calling `optimizer.step()`.

7. **LayerNorm** has two steps:
   - **Normalize**: compute mean and variance across `n_embd` (dim -1), per token independently → zero-mean, unit-variance
   - **Affine rescale**: learned γ `(n_embd,)` and β `(n_embd,)`, shared across all `B*T` tokens. Initialized to 1 and 0 (i.e. no-op at init).

In [1]:
import os
import math
import json
from dataclasses import dataclass
from typing import Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from transformers import GPT2LMHeadModel
from collections import defaultdict

from config import local_dir

In [2]:
device = "mps" if (hasattr(torch.backends, "mps") and torch.backends.mps.is_available()) else \
         "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

set_seed(12345)

device: mps


# 1. HF GPT-2 weight structure
Inspect param names and confirm 12 layers.

In [3]:
model_hf = GPT2LMHeadModel.from_pretrained("gpt2")
sd_hf = model_hf.state_dict()

# Group by category
categories = defaultdict(list)
for k in sd_hf:
    inner = k.removeprefix("transformer.")
    if inner.startswith("w"):     categories["emb"].append(inner)
    elif inner.startswith("h."):  categories["blocks"].append(inner)
    elif inner.startswith("ln"): categories["ln_f"].append(inner)
    else:                         categories["other"].append(k)

for cat, names in categories.items():
    print(f"\n{cat}:", names[:4], "..." if len(names) > 4 else "")

# Confirm each param appears in all 12 layers
per_layer = defaultdict(int)
for k in sd_hf:
    if k.startswith("transformer.h."):
        suffix = ".".join(k.split(".")[3:])
        per_layer[suffix] += 1

print("\nLayer count per param (should all be 12):")
for k, v in per_layer.items():
    print(f"  {k}: {v}")


emb: ['wte.weight', 'wpe.weight'] 

blocks: ['h.0.ln_1.weight', 'h.0.ln_1.bias', 'h.0.attn.c_attn.weight', 'h.0.attn.c_attn.bias'] ...

ln_f: ['ln_f.weight', 'ln_f.bias'] 

other: ['lm_head.weight'] 

Layer count per param (should all be 12):
  ln_1.weight: 12
  ln_1.bias: 12
  attn.c_attn.weight: 12
  attn.c_attn.bias: 12
  attn.c_proj.weight: 12
  attn.c_proj.bias: 12
  ln_2.weight: 12
  ln_2.bias: 12
  mlp.c_fc.weight: 12
  mlp.c_fc.bias: 12
  mlp.c_proj.weight: 12
  mlp.c_proj.bias: 12


# 2. Model

In [5]:
@dataclass
class GPTConfig:
    n_embd: int = 768
    n_head: int = 4
    n_layer: int = 2
    vocab_size: int = 50257   # tiktoken gpt2 vocab
    n_positions: int = 1024   # max sequence length

@dataclass
class ModelOutput:
    logits: torch.Tensor
    loss: torch.Tensor

In [ ]:
class CausalAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.c_attn = nn.Linear(config.n_embd, config.n_embd * 3)   # fused Q, K, V projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        B, T, C = x.shape
        d_head = C // self.n_head

        q, k, v = self.c_attn(x).split(C, dim=2)
        # reshape to (B, n_head, T, d_head) for multi-head attention
        q = q.contiguous().view(B, T, self.n_head, d_head).transpose(1, 2)
        k = k.contiguous().view(B, T, self.n_head, d_head).transpose(1, 2)
        v = v.contiguous().view(B, T, self.n_head, d_head).transpose(1, 2)

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)

# NOTE: below for details of F.scaled_dot_product_attention(q, k, v, is_causal=True)
# attn = q@k.transpose(-2,-1) /math.sqrt(d_head)
# mask = torch.triu(torch.ones(T, T, device = x.device, dtype = torch.bool), diagonal=1)
# attn = attn.masked_fill(mask, float('-inf'))

# attn = F.softmax(attn, dim = -1)
# out = attn @ v
#  -----

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(out)


class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, config.n_embd * 4)
        self.gelu   = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(config.n_embd * 4, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))


class Block(nn.Module):
    """Pre-norm transformer block: LN -> Attn -> residual, LN -> MLP -> residual."""

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp  = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [ ]:
class Transformer(nn.Module):

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.n_positions, config.n_embd),
            h   = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # weight tying: lm_head shares weights with token embedding
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)

    @property
    def device(self):
        return next(self.parameters()).device

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02
            # scale down residual projections by sqrt(2 * n_layer) to keep activations stable
            if hasattr(module, 'NANOGPT_SCALE_INIT'):  # fixed: was NOGPT_SCALE_INIT
                std *= (2 * self.config.n_layer) ** -0.5
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, tokens: torch.Tensor, targets: torch.Tensor = None) -> ModelOutput:
        B, T = tokens.shape
        pos = torch.arange(T, dtype=torch.long, device=tokens.device)
        x = self.transformer.wte(tokens) + self.transformer.wpe(pos)
        for block in self.transformer.h:
            x = block(x)
        logits = self.lm_head(self.transformer.ln_f(x))
        loss = F.cross_entropy(logits.view(B * T, -1), targets.view(-1)) if targets is not None else None
        return ModelOutput(logits=logits, loss=loss)

    @classmethod
    def from_pretrained(cls, model_type: str):
        """Load weights from a HuggingFace GPT-2 checkpoint."""
        configs = {
            'gpt2':        dict(n_layer=12, n_head=12, n_embd=768),
            'gpt2-medium': dict(n_layer=24, n_head=16, n_embd=1024),
            'gpt2-large':  dict(n_layer=36, n_head=20, n_embd=1280),
            'gpt2-xl':     dict(n_layer=48, n_head=25, n_embd=1600),  # fixed: was 7600
        }
        config = GPTConfig(**configs[model_type])
        model = cls(config)

        sd      = model.state_dict()
        sd_hf   = GPT2LMHeadModel.from_pretrained(model_type).state_dict()
        # HF stores Conv1D weights transposed relative to nn.Linear
        transposed = {'attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight'}

        for key in sd:
            with torch.no_grad():
                if any(key.endswith(t) for t in transposed):
                    assert sd[key].shape == sd_hf[key].shape[::-1]
                    sd[key].copy_(sd_hf[key].T)
                else:
                    assert sd[key].shape == sd_hf[key].shape, key
                    sd[key].copy_(sd_hf[key])
        return model

    def configure_optimizer(self, learning_rate: float, weight_decay: float):
        """AdamW with weight decay only on 2-D params (weight matrices)."""
        params = dict(self.named_parameters())
        decay     = [p for p in params.values() if p.requires_grad and p.dim() > 1]
        no_decay  = [p for p in params.values() if p.requires_grad and p.dim() <= 1]
        print(f"decay params: {sum(p.numel() for p in decay):,}  "
              f"no-decay params: {sum(p.numel() for p in no_decay):,}")
        return torch.optim.AdamW(
            [{'params': decay, 'weight_decay': weight_decay},
             {'params': no_decay, 'weight_decay': 0.0}],
            lr=learning_rate, betas=(0.9, 0.95), eps=1e-8,
        )


# --- smoke test ---
enc    = tiktoken.get_encoding('gpt2')
config = GPTConfig(n_layer=2)  # small for testing
model  = Transformer(config).to(device)
model.configure_optimizer(1e-4, 0.1)

# 3. DataLoader

In [ ]:
def _load_tokens(path: str) -> torch.Tensor:
    return torch.tensor(np.load(path), dtype=torch.long)


class DataLoader:

    def __init__(self, B: int, T: int, data_dir: str, split: str):
        self.B, self.T = B, T
        self.shards = sorted(
            os.path.join(data_dir, f) for f in os.listdir(data_dir) if split in f
        )
        self.reset()

    def reset(self):
        self.shard_idx = 0
        self.pos       = 0
        self.tokens    = _load_tokens(self.shards[0])

    def next_batch(self) -> Tuple[torch.Tensor, torch.Tensor]:
        n = self.B * self.T
        chunk = self.tokens[self.pos : self.pos + n + 1]
        x, y  = chunk[:-1].view(self.B, self.T), chunk[1:].view(self.B, self.T)
        self.pos += n
        if self.pos + n + 1 > len(self.tokens):
            self.shard_idx = (self.shard_idx + 1) % len(self.shards)
            self.tokens    = _load_tokens(self.shards[self.shard_idx])
            self.pos       = 0
        return x, y


B, T = 64, 256
train_loader = DataLoader(B, T, local_dir, 'train')
val_loader   = DataLoader(B, T, local_dir, 'val')

# gradient accumulation: simulate 0.5M token batches
n_tokens_per_batch = 524288
assert n_tokens_per_batch % (B * T) == 0
grad_accum_steps = n_tokens_per_batch // (B * T)
print(f"grad_accum_steps: {grad_accum_steps}")

# 4. Evaluation

In [ ]:
@torch.no_grad()
def get_val_loss(model: nn.Module, val_steps: int = 20) -> float:
    model.eval()
    total = 0.0
    for _ in range(val_steps):
        x, y = val_loader.next_batch()
        total += model(x.to(model.device), y.to(model.device)).loss.item()
    val_loss = total / val_steps
    print(f"val loss: {val_loss:.4f}")
    return val_loss


def save_checkpoint(model: nn.Module, out_dir: str, step: int, val_loss: float):
    os.makedirs(out_dir, exist_ok=True)
    torch.save({'model': model.state_dict(), 'config': model.config,
                'step': step, 'val_loss': val_loss},
               os.path.join(out_dir, f"model_{step:05d}.pt"))

In [ ]:
@torch.no_grad()
def complete_sentence(
    model, encoder, text: str,
    n_examples: int = 4, max_new_tokens: int = 30,
    top_k: int = 20, temperature: float = 0.6,
):
    was_training = model.training
    model.eval()

    tokens = torch.tensor(encoder.encode_ordinary(text)).unsqueeze(0)
    tokens = tokens.repeat(n_examples, 1).to(model.device)  # (n_examples, T)

    for _ in range(max_new_tokens):
        logits = model(tokens).logits[:, -1, :]          # (B, vocab)
        probs  = F.softmax(logits / temperature, dim=-1)
        top_p, top_idx = torch.topk(probs, k=top_k, dim=-1)  # (B, k)
        chosen = torch.multinomial(top_p, 1)                  # (B, 1)
        next_tok = torch.gather(top_idx, -1, chosen)          # (B, 1)
        tokens = torch.cat([tokens, next_tok], dim=-1)

    for i in range(n_examples):
        toks = tokens[i].tolist()
        if encoder.eot_token in toks:
            toks = toks[:toks.index(encoder.eot_token)]
        print(encoder.decode(toks))  # fixed: was always printing example i=last

    if was_training:
        model.train()

complete_sentence(model, enc, "What makes a person resilient")

In [ ]:
def iter_hellaswag(split: str = 'val', limit: int = 50):
    with open(f"hellaswag/hellaswag_{split}.jsonl") as f:
        for i, line in enumerate(f):
            if i >= limit:
                break
            yield json.loads(line)


def render_example(example: dict, encoder) -> Tuple[torch.Tensor, torch.Tensor, int]:
    ctx_tokens = encoder.encode_ordinary(example['ctx'])
    label      = int(example['label'])
    max_len    = 0
    all_tokens, all_masks = [], []

    for ending in example['endings']:
        end_toks = encoder.encode_ordinary(' ' + ending)
        row = ctx_tokens + end_toks
        all_tokens.append(row)
        all_masks.append([0] * len(ctx_tokens) + [1] * len(end_toks))
        max_len = max(max_len, len(row))

    pad_tokens = torch.zeros((4, max_len), dtype=torch.long)
    pad_masks  = torch.zeros((4, max_len), dtype=torch.long)
    for i in range(4):
        L = len(all_tokens[i])
        pad_tokens[i, :L] = torch.tensor(all_tokens[i])
        pad_masks[i,  :L] = torch.tensor(all_masks[i])

    return pad_tokens, pad_masks, label


@torch.no_grad()
def eval_hellaswag(model, encoder, split: str = 'val', limit: int = 50):
    was_training = model.training
    model.eval()
    correct_norm = correct = total = 0

    for example in iter_hellaswag(split, limit):
        tokens, masks, label = render_example(example, encoder)
        x = tokens[:, :-1].to(model.device)
        y = tokens[:, 1:].contiguous().to(model.device)
        m = masks[:, 1:].to(model.device)   # shift mask to align with targets

        logits = model(x).logits
        losses = F.cross_entropy(
            logits.view(-1, logits.size(-1)), y.view(-1), reduction='none'
        ).view(x.size(0), -1)               # (4, T-1)

        total_loss = (losses * m).sum(dim=-1)
        avg_loss   = total_loss / m.sum(dim=-1)

        correct      += total_loss.argmin().item() == label
        correct_norm += avg_loss.argmin().item()   == label
        total        += 1

    print(f"HellaSwag ({total} examples) — avg-loss acc: {correct_norm}/{total}, "
          f"total-loss acc: {correct}/{total}")
    if was_training:
        model.train()
    return correct_norm, correct, total

# 5. Training

In [ ]:
# Cosine LR schedule with linear warmup
max_lr, min_lr        = 6e-4, 6e-5
warm_up_steps         = 715
max_steps             = 19073   # ~1 epoch at 10B tokens, 0.5M token batches

def get_lr(step: int) -> float:
    if step < warm_up_steps:
        return max_lr * (step + 1) / warm_up_steps
    if step > max_steps:
        return min_lr
    ratio = (step - warm_up_steps) / (max_steps - warm_up_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return min_lr + coeff * (max_lr - min_lr)

In [ ]:
optimizer     = model.configure_optimizer(learning_rate=max_lr, weight_decay=0.1)
max_train_steps = 10

for step in range(max_train_steps):

    # --- eval every 2 steps ---
    if step % 2 == 0:
        print(f"\n=== step {step} ===")
        val_loss = get_val_loss(model)
        save_checkpoint(model, 'checkpoints', step, val_loss)
        complete_sentence(model, enc, 'What makes a person resilient')
        eval_hellaswag(model, enc)

    # --- train step with gradient accumulation ---
    model.train()
    optimizer.zero_grad()
    accum_loss = 0.0

    for _ in range(grad_accum_steps):
        x, y = train_loader.next_batch()
        x, y = x.to(device), y.to(device)
        loss  = model(x, y).loss / grad_accum_steps
        accum_loss += loss.detach().item()
        loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    for pg in optimizer.param_groups:
        pg['lr'] = get_lr(step)
    optimizer.step()

    print(f"step {step}: train loss {accum_loss:.4f}  lr {get_lr(step):.2e}")